# Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')
import gensim
from gensim import corpora
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import re

# Upload Data - Excerpts of 3 Sequential phrases

In [ ]:
output_path = '/Users/vi3680mu/Library/CloudStorage/GoogleDrive-murarosilva@gmail.com/My Drive/[1] Lund University/[1] Research Projects/Digital Public Infrastructures/Data/Results_jan2026/'
excerpts_df = pd.read_csv(output_path + 'excerpts_3sequential_may2025.csv')
excerpts_df

In [ ]:
# Remove excerpts with less than 300 characters
excerpts_df = excerpts_df[excerpts_df['sequential_sentences_excerpts'].apply(len) > 300]
len(excerpts_df)

# Data Cleaning

In [ ]:
extended_stopwords = {
    # --- OCR / segmentation junk ---
    "cid3", "cid3cid3", "cid3 cid3", "cid3 cid3cid3", "cid3cid3 cid3cid3",
    "acific", "artnership", "apanese", "ndian",
    "lation pursuant", "legis lation", "ther specify", "xpolicies",
    "redistributionimf working", "redistributionimf world",
    "inside iew", "ocasional", "documento ocasional",
    "big ech", "exploration characterize", "explore characterize",
    "ykey", "ykey consideration",
    "lin lin", "lin zero", "zero lin",
    "period percent", "figure period",
    "zero zero", "jan zero", "tabjul", "tabjul jan",
    "19 100", "covax covax", "lin", "zero", "comma", "period"

    # --- report admin / production artefacts ---
    "00", "pl", "v2", "gfc2020", "gfc2020 v2",
    "vh", "aow", "oc", "pbm", "daps", "s3i",
    "g584", "gpw 14", "gpw14",
    "fy23", "fy2023", "fy2024", "fy2025",
    "ida 20", "2022 2025", "2025 2028",
    "2030 2035", "2035 2040", "2040 2045", "2045",
    "dp 202", "md en", "cr", "dpga", "iui",
    "iti", "nti", "debi", "itida",
    "pgt", "apg", "apgs", "dnc", "dpit", "pnc",
    "ecs"
}

def basic_cleaning(text):
	# Remove headers/footers, extra spaces
	cleaned_text = re.sub(r'\n\s*\n', '\n\n', text)  # Consolidate blank lines
	cleaned_text = re.sub(r'\s*\d+\s*\n', '\n', cleaned_text)  # Remove single-line page numbers
	cleaned_text = re.sub(r'\(cid:\d+\)', '', cleaned_text)  # Remove PDFMiner specific tags

	# Word Analysis (remove noise)
	words = re.findall(r'\b\w+\b', cleaned_text.lower())
	stop_words = set(["tab", "https", "www"]) | extended_stopwords
	filtered_words = [word for word in words if word not in stop_words and len(word) > 1]
	return ' '.join(filtered_words)

In [ ]:
excerpts_df['cleaned_excerpts'] = excerpts_df['sequential_sentences_excerpts'].apply(basic_cleaning)
excerpts_df

In [ ]:
#excerpts_df.to_csv(output_path + 'excerpts_cleaned_6jan2026.csv', index=False, encoding='utf-8')
excerpts_df = pd.read_csv(output_path + 'excerpts_cleaned_6jan2026.csv')

# BERTopics

## BERTopic for identifying DPI excerpts

In [ ]:
# Step 1: Prepare the data
documents = excerpts_df['cleaned_excerpts'].tolist()

In [ ]:
# Step 2: Create a BERTopic model
vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words="english", min_df=5)
topic_model = BERTopic(vectorizer_model=vectorizer_model, language = "english")

# Step 3: Fit the model on the documents
topics, probabilities = topic_model.fit_transform(documents)

In [ ]:
#topic_model.save(output_path + "bertopic_cleaned_6jan2026")
# Uncomment to reload the model
topic_model = BERTopic.load(output_path + "bertopic_cleaned_6jan2026")

In [ ]:
topic_model.get_topic_info()

In [ ]:
# Reduce the number of topics to 1000
topic_model.reduce_topics(documents, nr_topics=1000)

In [ ]:
#topic_model.save(output_path + "bertopic_cleaned_reduced_6jan2026")
topic_model = BERTopic.load(output_path + "bertopic_cleaned_reduced_6jan2026")

In [ ]:
# Topic info reduced
topic_info = topic_model.get_topic_info()
topic_info

In [ ]:
# Identifying 20 topics related to "digital public infrastructure" or "dpi"
topics_found, simils = topic_model.find_topics(["digital public infrastructure", "dpi"], top_n=20)
# Show topics in the order of similarity
topic_info[topic_info['Topic'].isin(topics_found)].set_index('Topic').loc[topics_found]

In [ ]:
excerpts_df['Topic']=topic_model.get_document_info(documents).Topic

## BERTopic for identifying discourse about DPI

In [ ]:
# Filter excerpts_df to only documents classified in topics_found
excerpts_dpi = excerpts_df[excerpts_df['Topic'].isin(topics_found)]

In [ ]:
#excerpts_dpi.to_csv(output_path + 'excerpts_dpi_6jan2026.csv', index=False, encoding='utf-8')
excerpts_dpi = pd.read_csv(output_path + 'excerpts_dpi_6jan2026.csv')

In [ ]:
# Prepare the filtered documents
filtered_documents = excerpts_dpi['cleaned_excerpts'].tolist()

In [ ]:
# Rerun BERTopic on the filtered documents
filtered_topic_model = BERTopic(vectorizer_model=CountVectorizer(ngram_range=(1, 3), stop_words="english", min_df=5), language="english")


filtered_topics, filtered_probabilities = filtered_topic_model.fit_transform(filtered_documents)

In [ ]:
#filtered_topic_model.save(output_path + "bertopic_dpi_6jan2026")
filtered_topic_model = BERTopic.load(output_path + "bertopic_dpi_6jan2026")

In [ ]:
filtered_topic_model.get_topic_info()

In [ ]:
# Step 5: Visualize the topics
# Visualize Topics
fig_topics = filtered_topic_model.visualize_topics()
fig_topics.show()

# Visualize Topic Hierarchy
# fig_hierarchy = filtered_topic_model.visualize_hierarchy()
# fig_hierarchy.show()

# Visualize Topic Similarity
fig_heatmap = filtered_topic_model.visualize_heatmap()
fig_heatmap.show()

# Visualize Topic Distribution
fig_barchart = filtered_topic_model.visualize_barchart()
fig_barchart.show()

In [ ]:
# Reduce topics to 100
filtered_topic_model.reduce_topics(filtered_documents, nr_topics=100)

In [ ]:
filtered_topic_info = filtered_topic_model.get_topic_info()
filtered_topic_info

In [ ]:
filtered_topic_info.to_excel(output_path + 'bertopic_dpi_reduced_7jan2026.xlsx', index=False)

In [ ]:
filtered_topic_model.save(output_path + "bertopic_dpi_reduced_6jan2026")
#filtered_topic_model = BERTopic.load(output_path + "bertopic_dpi_reduced_6jan2026")

# NOT IN USE BELOW HERE
## Match Representation tokens with keywords of interest (Promoting and safeguarding)

In [ ]:
topic_info = filtered_topic_model.get_topic_info()

In [ ]:
promoting_dpi_keywords = {
    "technologies": ["artificial intelligence", "ai", "Big Data", "smart city", "digital ID", "blockchain", "cloud", "cryptocurrencies", "CBDC", "agtech", "fintech"],
    "actors": ["startups", "innovation hubs", "investors", "banks"],
    "processes": ["digital transformation", "sustainable transformation", "digital government", "open finance", "investments", "asset management", "automation", "infrastructural development"],
    "attributes": ["public goods", "interoperability"],
    "results": ["innovation", "economic growth", "added value", "economic security", "trade", "fiscal credibility", "wealth", "patents", "growth", "productivity", "employment", "competitiveness", "technology diffusion"]
}

safeguarding_dpi_keywords = {
    "technologies": ["cybersecurity", "surveillance economy", "safeguarding devices"],
    "actors": ["communities", "diversity group", "social council", "organisation activists", "society organisations"],
    "processes": ["social protection", "data protection", "regulation", "citizen engagement", "self determination", "privacy respect", "accountability", "corruption control", "redressal mechanism", "oversight", "online censorship"],
    "attributes": ["just", "trust", "safe", "universal", "democratic", "civic", "accountable", "agency centred", "biased"],
    "results": ["empowerment", "equality", "accessibility", "justice", "rights", "freedom", "confidence", "transparency", "integrity", "fairness", "social responsibility"]
}

In [ ]:
# Complete list of keywords for Promoting DPI
promoting_dpi_all_keywords = [
    keyword for sublist in promoting_dpi_keywords.values() for keyword in sublist
]

# Complete list of keywords for Safeguarding DPI
safeguarding_dpi_all_keywords = [
    keyword for sublist in safeguarding_dpi_keywords.values() for keyword in sublist
]

# Variables with sublists for each part of Promoting DPI
promoting_technologies = promoting_dpi_keywords["technologies"]
promoting_actors = promoting_dpi_keywords["actors"]
promoting_processes = promoting_dpi_keywords["processes"]
promoting_attributes = promoting_dpi_keywords["attributes"]
promoting_results = promoting_dpi_keywords["results"]

# Variables with sublists for each part of Safeguarding DPI
safeguarding_technologies = safeguarding_dpi_keywords["technologies"]
safeguarding_actors = safeguarding_dpi_keywords["actors"]
safeguarding_processes = safeguarding_dpi_keywords["processes"]
safeguarding_attributes = safeguarding_dpi_keywords["attributes"]
safeguarding_results = safeguarding_dpi_keywords["results"]

# Print the complete lists for verification
print("Promoting DPI Keywords:", promoting_dpi_all_keywords)
print("Safeguarding DPI Keywords:", safeguarding_dpi_all_keywords)

In [ ]:
topic_info.Representation

In [ ]:
# Function to check if any keyword from the seed list is present in the Representation list (case-insensitive)
def contains_keyword(representation, seed_list):
    return int(any(keyword.lower() in word.lower() for word in representation for keyword in seed_list))

In [ ]:
topic_info['Promoting'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_dpi_all_keywords))
topic_info['Safeguarding'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_dpi_all_keywords))
topic_info['Promoting_technologies'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_technologies))
topic_info['Promoting_actors'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_actors))
topic_info['Promoting_processes'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_processes))
topic_info['Promoting_attributes'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_attributes))
topic_info['Promoting_results'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_results))
topic_info['Safeguarding_technologies'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_technologies))
topic_info['Safeguarding_actors'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_actors))
topic_info['Safeguarding_processes'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_processes))
topic_info['Safeguarding_attributes'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_attributes))
topic_info['Safeguarding_results'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_results))

In [ ]:
topic_info[topic_info['Safeguarding'] == 1]

In [ ]:
excerpts_df['topics'] = docs_info.Topic.to_list()
excerpts_df